## Step 2 — Dataset Selection

Use a combined dataset with the three classes. A practical student-friendly approach:

- Start with a public fresh/rotten dataset (Kaggle)
- Add a curated `slightly_spoiled` class using web images (Chapter 2 style)
- Clean mistakes before training

Target folder layout:

```text
data/food_freshness/
├── fresh/
├── slightly_spoiled/
└── rotten/
```

Keep classes reasonably balanced. If one class has too few examples, your model may over-predict the larger classes.

In [ ]:
from pathlib import Path

from fastai.vision.all import *

set_seed(42)
DATA_PATH = Path('data/food_freshness')
DATA_PATH

## Step 3 — Data Loading with DataBlock

Before building `DataLoaders`, we verify image files and remove corrupted ones.

In [ ]:
all_images = get_image_files(DATA_PATH)
failed = verify_images(all_images)
failed

In [ ]:
for img_path in failed:
    img_path.unlink(missing_ok=True)

print(f'Removed {len(failed)} corrupted files.')

### DataBlock definition (core Chapter 3 concept)

- `ImageBlock`: input is an image
- `CategoryBlock`: output is a categorical label
- `get_items=get_image_files`: collect image files recursively
- `get_y=parent_label`: label comes from parent folder name
- `splitter=RandomSplitter(...)`: train/valid split
- `item_tfms=Resize(224)`: standard input size

In [ ]:
food_block = DataBlock(
    blocks=(ImageBlock, CategoryBlock),
    get_items=get_image_files,
    get_y=parent_label,
    splitter=RandomSplitter(valid_pct=0.2, seed=42),
    item_tfms=Resize(224),
)

dls = food_block.dataloaders(DATA_PATH, bs=32)
dls.show_batch(max_n=9, figsize=(8, 8))

In [ ]:
dls.vocab

In [ ]:
from collections import Counter

label_counts = Counter(parent_label(p) for p in get_image_files(DATA_PATH))
label_counts

## Step 4 — Data Augmentation

`aug_transforms()` creates realistic variations such as rotation, zoom, lighting changes, warping, and flips. This improves generalization by helping the model learn robust features, not just memorized pixels.

In [ ]:
food_block_aug = DataBlock(
    blocks=(ImageBlock, CategoryBlock),
    get_items=get_image_files,
    get_y=parent_label,
    splitter=RandomSplitter(valid_pct=0.2, seed=42),
    item_tfms=Resize(224),
    batch_tfms=[
        *aug_transforms(size=224, min_scale=0.75),
        Normalize.from_stats(*imagenet_stats),
    ],
)

dls = food_block_aug.dataloaders(DATA_PATH, bs=32)
dls.show_batch(max_n=9, figsize=(8, 8))

## Step 5 — Build the Model with `vision_learner()`

We use `resnet34` (beginner-friendly, strong baseline) and transfer learning with pretrained ImageNet weights.

- **Body**: pretrained ResNet feature extractor
- **Head**: task-specific classifier for 3 classes
- **Metrics**: `accuracy` and `error_rate`

In [ ]:
learn = vision_learner(
    dls,
    resnet34,
    metrics=[accuracy, error_rate],
)
learn.summary()

## Step 6 — Learning Rate Finder

`lr_find()` suggests promising learning rates. Choose a value near the steepest downward part before loss gets unstable.

In [ ]:
lr_suggestion = learn.lr_find()
lr_suggestion

## Step 7 — Training with `fine_tune()`

`fine_tune()` first trains the new head with the pretrained body frozen, then unfreezes and trains all layers with discriminative learning rates.

Start with short training, then extend if validation improves.

In [ ]:
# First quick pass
learn.fine_tune(5, base_lr=1e-3)

In [ ]:
# Longer run for better accuracy (adjust base_lr using lr_find result)
learn.fine_tune(10, base_lr=3e-4)

## Step 8 — Evaluation and Interpretation

We inspect confusion matrix and top losses to understand mistakes and dataset weaknesses.

In [ ]:
interp = ClassificationInterpretation.from_learner(learn)
interp.plot_confusion_matrix(figsize=(6, 6), dpi=100)

In [ ]:
interp.plot_top_losses(9, nrows=3)

In [ ]:
# Try one sample prediction from validation set
img_path = dls.valid_ds.items[0]
pred_class, pred_idx, probs = learn.predict(img_path)
print(f'Image: {img_path.name}')
print(f'Prediction: {pred_class}')
print({dls.vocab[i]: float(probs[i]) for i in range(len(probs))})

### Optional: `ImageClassifierCleaner`

If your labels are noisy, this FastAI widget helps spot likely mislabels and duplicates. Use this before final training for better quality.

In a notebook UI environment, you can run:

```python
cleaner = ImageClassifierCleaner(learn)
cleaner
```

Then move/delete problematic images and recreate `DataLoaders`.

## Step 9 — Export the Learner

FastAI saves preprocessing + model together via `export()`. This makes deployment easier because inference uses the same transforms and label vocabulary.

In [ ]:
model_dir = Path('model')
model_dir.mkdir(parents=True, exist_ok=True)
learn.export(model_dir / 'export.pkl')
print('Exported model to model/export.pkl')

## Step 10 & 11 — Deployment

Next steps:

1. Run `python app.py` locally and test uploads.
2. Push repository to GitHub.
3. Create a Gradio Space and deploy with `app.py`, `requirements.txt`, and `model/export.pkl`.

This completes a full Chapters 1-4 style project pipeline from data to deployment.